# MNIST demo with CREST


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

#### Load MNIST Datasets


In [3]:
from crest import DataServer

# Load datasets from server
ds_train, ds_valid, ds_test, ds_pred = DataServer.load('mnist')

# Get input/output labels
[INP, OUT] = ds_train.labels

# Num classes
n_classes = 10
patch_size = 28

2024-02-02 15:07:27.661996: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph
from crest.model.KerasNode import KerasNode
from crest.model.TensorSpec import TensorSpec

layers = [
    KerasNode(Lambda(lambda x: tf.cast(x, tf.float32)), name='layer_0', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28])}),        # Cast dtype
    KerasNode(Lambda(lambda x: tf.expand_dims(x, axis=-1)), name='layer_1', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28, 1])}),  # Add channel dimension
    KerasNode(Conv2D(10, 3, padding='same', activation='relu'), name='layer_2', inputs={INP: TensorSpec(
        [None, 28, 28, 1])}, outputs={OUT: TensorSpec([None, 28, 28, 10])}),  # Convolutional layer
    KerasNode(Flatten(), name='layer_3', inputs={INP: TensorSpec(
        [None, 28, 28, 10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Flatten
    KerasNode(Dropout(0.3), name='layer_4', inputs={INP: TensorSpec(
        [None, 28*28*10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Dropout
    KerasNode(Dense(n_classes, activation='softmax'), name='layer_5', inputs={
              INP: TensorSpec([None, 28*28*10])}, outputs={OUT: TensorSpec([None, 10])})  # Output layer
]

for i in layers[:-1]:
    i.rename_io(outputs_map={OUT: INP})

graph = HierarchalTensorGraph(
    name='MNIST',
    inputs={INP: TensorSpec([None, patch_size, patch_size])},
    outputs={OUT: TensorSpec([None, 10])}
)

graph.add_edge('input', layers[0])

for i, layer in enumerate(layers[1:]):
    graph.add_edge(layers[i], layers[i+1])

graph.add_edge(layers[-1], 'output')

2024-02-02 15:07:49.511338: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


#### Create the HierarchalTensorGraph


In [5]:
# from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
# import tensorflow as tf
# from crest import HierarchalTensorGraph as HTG
# from crest.model.TensorSpec import TensorSpec

# def crest_layer(layer):
#     """ Wrap a tensorflow layer in a HierarchalTensorGraph """
#     graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
#     #set inputs/outputs
#     graph.inputs = {INP:None}
#     graph.outputs = {OUT:None}
#     return graph

# # Define the tensorflow layers, and convert to HTG objects
# layers = [crest_layer(layer) for layer in [
#     Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
#     Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
#     Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
#     Flatten(),                                        # 28x28x10 -> 7840
#     Dropout(0.3),                                     # 30% of data dropped
#     Dense(n_classes, activation='softmax'),           # Class probabilities
# ]]

# # Rename outputs to use as inputs (except the last layer)
# for i in layers[:-1]:
#     i.rename_io(outputs_map={OUT:INP})

# # Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
# graph = HTG(
#     name='MNIST',
#     inputs={INP : TensorSpec([None,patch_size, patch_size])},
#     outputs={OUT : TensorSpec([None,10])}
# )
# print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

#### Build a model using the HTG


In [6]:
from crest import Model

# Define a custom metric if desired
def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)


# Set build options
build_kwargs = {
    'optimizer': 'Adam',
    'loss': 'sparse_categorical_crossentropy',
    'metrics': ['sparse_categorical_accuracy', custom_metric]
}

# Create anad build a model using the HTG
model = Model(graph)
model.build(**build_kwargs)

/Users/draghun1/Documents/clean/crest/crest/model/Model.py:82: UserWarning: Use Model.compile instead of Model.build
  warnings.warn('Use Model.compile instead of Model.build')


#### Train the model


In [7]:
# Create the training and validation batchers
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size': 200,            # 200 samples in each batch
    'features': [[INP], [OUT]],  # [{'image': ...}, {'class': ...}]
    'repeat': True,           # Yield indefinitely
    'workers': 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)

with batch_train as train, batch_valid as valid:
    model.fit(train, **{
        'validation_data': valid,
        'steps_per_epoch': 250,  # 50k samples / 200 samples per batch
        'validation_steps': 50,  # 10k samples / 200 samples per batch
        'epochs': 5,
        'verbose': 1
    })

Epoch 1/5
250/250 [==============================] - 11s 41ms/step - loss: 2.9003 - sparse_categorical_accuracy: 0.8805 - custom_metric: 27.4167 - val_loss: 0.3453 - val_sparse_categorical_accuracy: 0.9760 - val_custom_metric: 27.4874
Epoch 2/5
250/250 [==============================] - 11s 44ms/step - loss: 0.3983 - sparse_categorical_accuracy: 0.9442 - custom_metric: 27.3804 - val_loss: 0.2080 - val_sparse_categorical_accuracy: 0.9700 - val_custom_metric: 27.4868
Epoch 3/5
250/250 [==============================] - 11s 44ms/step - loss: 0.1985 - sparse_categorical_accuracy: 0.9605 - custom_metric: 27.3242 - val_loss: 0.1782 - val_sparse_categorical_accuracy: 0.9750 - val_custom_metric: 27.4868
Epoch 4/5
250/250 [==============================] - 11s 44ms/step - loss: 0.1370 - sparse_categorical_accuracy: 0.9666 - custom_metric: 27.4247 - val_loss: 0.1543 - val_sparse_categorical_accuracy: 0.9780 - val_custom_metric: 27.4868
Epoch 5/5
250/250 [==============================] - 11s 44m

#### Evaluate the model


In [8]:
# Evaluate the model on the test set
batch_test = Batcher(ds_test, **batch_kwargs)
evaluate_kwargs = {
    'verbose': 1,
    'return_dict': True
}

with batch_test as batcher:
    res = model.evaluate(batcher, verbose=1, return_dict=1)

1/1 [==============================] - 0s 38ms/step - loss: 0.0634 - sparse_categorical_accuracy: 0.9800 - custom_metric: 26.3266


Accuracy on the full test set should be > ~97%.


### Make predictions


In [9]:
# Create batcher for predictions
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size': 200,
    'features': [INP],
    'repeat': True,
    'workers': 0,
    'shuffle': False
})

# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size': 200,
    'verbose': False,
    'steps': 10000//200
}

# Generate predictions
with batch_pred as batcher:
    model_pred_batch = model.predict(
        batch_pred, **pred_kwargs)  # Using a batcher
    model_pred_dataset = model.predict(
        ds_pred, **pred_kwargs)   # Using a dataset
    print('Difference:',
          (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


#### Adding additional output (e.g., lat and lon) when making a predictions


In [10]:
# Create inputs with additional features lat and lon
import numpy as np
from crest.data.loading import StructuredDataset
x = ds_pred.data[0]
lat = np.arange(len(x), dtype=float)
lon = np.arange(len(x), dtype=float)
x_aux = (lat, lon, x)
ds_aux = StructuredDataset(*x_aux, labels=['lat', 'lon', INP])

# Specify additional output as coords when making predictions
model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

{'class': array([[4.68896211e-16, 1.17742229e-20, 5.16652389e-12, ...,
         1.00000000e+00, 5.23147172e-13, 7.58886096e-12],
        [1.69176964e-11, 1.23916744e-09, 9.99253690e-01, ...,
         2.65149718e-17, 4.37146812e-11, 6.13982378e-18],
        [1.71118977e-08, 9.99827981e-01, 7.36668085e-07, ...,
         2.72790061e-07, 1.69130712e-04, 1.75884515e-10],
        ...,
        [3.26602504e-17, 1.89946359e-14, 4.63687111e-20, ...,
         3.55084434e-11, 1.29283140e-10, 1.56246238e-10],
        [4.53009754e-14, 9.62043610e-16, 2.79668442e-17, ...,
         1.28098560e-19, 2.97941801e-06, 6.43613288e-11],
        [7.95349064e-10, 1.03265405e-13, 1.67296108e-11, ...,
         5.17330212e-18, 1.34685015e-13, 2.81406150e-17]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}

In [11]:
print('Elapsed time = ', time.time() - start_time, '[s]')

Elapsed time =  90.3307147026062 [s]


model save and load

In [16]:
model.save()

In [19]:
loaded_model = model.load('crest_cache')

In [23]:
# check that loaded model performs the same as original
result_loaded = loaded_model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)
result = model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

print('Difference:', (result['class'] - result_loaded['class']).sum())

Difference: 0.0
